[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [ ]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [ ]:
data_frame = pd.read_csv("2.1.2.minecraft_100x100.csv")

#### Dealing with null values

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method call returns the null values in any column.

In [ ]:
pd.set_option("display.max_rows", None)
data_frame.isnull().sum()

If you have null data there are many ways to deal with the empty/null values. These are the two most common approaches.
1. Remove any row with a null value with a `dropna()` method call.
2. Replace missing values with another value with a `fillna()` method call. Generally, we use mean value for numerical columns because it may cause minimal changes in your mathematical analysis while maintaining the original size of the data.

Students should reflect why this example removes the null 'SEX' but replacing the mean 'Target'?

In [ ]:
# No Null values

# data_frame = data_frame.dropna(subset=['SEX'])
# data_frame.isnull().sum()

In [ ]:
# No Null values

# data_frame['Target'] = data_frame['Target'].fillna(data_frame['Target'].mean())
# data_frame.isnull().sum()

#### Replace data

In order to improve the readibility and usability of my targets, I am going to remove the common prefix from every value.

In [ ]:
data_frame['dominant_biome'] = data_frame['dominant_biome'].apply(lambda x: x.replace('minecraft:', ''))
data_frame['dominant_biome'].head()

Then, I'll do the same with my features (to improve load time or readability)

In [ ]:
data_frame.columns = data_frame.columns.str.replace("minecraft:", "", regex=False)
data_frame.head()

#### Drop/combine unecessary targets

(for the purpose of simplifying the algorithm and preventing overfitting)

In [ ]:
data_frame['dominant_biome'].unique()

In [ ]:
data_frame['dominant_biome'] = data_frame['dominant_biome'].apply(
    lambda biome: 'ocean' if 'ocean' in biome else biome
)
data_frame["dominant_biome"] = data_frame["dominant_biome"].apply(
    lambda biome: "taiga" if "taiga" in biome else biome
)
data_frame["dominant_biome"] = data_frame["dominant_biome"].apply(
    lambda biome: "birch_forest" if "birch_forest" in biome else biome
)
data_frame["dominant_biome"].unique()

#### Drop unecessary features

Many of my features are not necessary for the algorithms prediction, either being very rare (only in one or two chunks), or are in every single chunk no matter what (e.g. bedrock).

In [ ]:
df = pd.read_csv("2.1.2.minecraft_100x100.csv")
zero_counts = (df == 0).sum(axis=0)
zero_summary = pd.DataFrame({"Column": df.columns, "Zero Count": zero_counts.values})
zero_summary = zero_summary.sort_values("Zero Count", ascending=True)
print(zero_summary.to_string(index=False))

I have attempted to filter features by the amount of 0 values to determine their importance, but I have decided that I will need to manually select which features to keep based on my domain knowledge.

In [ ]:
# chunk_x, chunk_z, dominant_biome, air, dirt, water, short_grass, grass_block, lava, sand, tall_seagrass, oak_leaves, birch_leaves, oak_log, tall_grass, birch_log, kelp, kelp_plant, oak_planks, oak_fence, dark_oak_log, dark_oak_leaves, spruce_leaves, spruce_log, bush, sandstone, brown_mushroom, dandelion, podzol, red_mushroom_block, mushroom_stem, poppy, coarse_dirt, oxeye_daisy, cornflower, azure_bluet, emerald_ore, lily_pad, barrel, sugar_cane, red_mushroom, lilac, lily_of_the_valley, rose_bush, peony, sweet_berry_bush, bee_nest, pumpkin, snow, acacia_leaves, acacia_log, blue_orchid

columns_to_keep = [
    "chunk_x",
    "chunk_z",
    "dominant_biome",
    "air",
    "dirt",
    "water",
    "short_grass",
    "grass_block",
    "lava",
    "sand",
    "tall_seagrass",
    "oak_leaves",
    "birch_leaves",
    "oak_log",
    "tall_grass",
    "birch_log",
    "kelp",
    "kelp_plant",
    "oak_planks",
    "oak_fence",
    "dark_oak_log",
    "dark_oak_leaves",
    "spruce_leaves",
    "spruce_log",
    "bush",
    "sandstone",
    "brown_mushroom",
    "dandelion",
    "podzol",
    "red_mushroom_block",
    "mushroom_stem",
    "poppy",
    "coarse_dirt",
    "oxeye_daisy",
    "cornflower",
    "azure_bluet",
    "emerald_ore",
    "lily_pad",
    "barrel",
    "sugar_cane",
    "red_mushroom",
    "lilac",
    "lily_of_the_valley",
    "rose_bush",
    "peony",
    "sweet_berry_bush",
    "bee_nest",
    "pumpkin",
    "snow",
    "acacia_leaves",
    "acacia_log",
    "blue_orchid",
]

data_frame = data_frame[columns_to_keep]
data_frame.head()

#### Remove outliers

Outliers can skew your analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [ ]:
# get the inter-quartile range on all biomes blocks
# Outliers are negligible and may have to be adressed after cutting down features

blocks = [
    'air', 'water'
]
for block in blocks:
    print(data_frame[block].describe())
    Q1 = data_frame[block].quantile(0.10)
    Q3 = data_frame[block].quantile(0.90)
    IQR = Q3 - Q1
    print(f'Outliers are a {block} count above {Q3 + IQR} or below {Q1 - IQR}')

In [ ]:
# Outliers are not being filtered as they are negligible

# data_frame = data_frame[(data_frame['BP'] >= Q1 - 1.5 * IQR) & (data_frame['BP'] <= Q3 + 1.5 * IQR)]
# print(data_frame['BP'].describe())

#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [ ]:
# All features are on the exact same scale already, no need for feature scaling

# scale_feature = 'BP'
# #the minimum value with space for outliers
# MIN_BP = 55
# #the maximum value with space for outliers
# MAX_BP = 140
# #scale features
# data_frame[scale_feature] = [(X - MIN_BP) / (MAX_BP - MIN_BP) for X in data_frame[scale_feature]]
# data_frame.describe()

> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [ ]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False)